#Introduction to GPU Acceleration
### 🔍 Why Use GPUs?

GPUs are optimized for large-scale parallel computation, making them ideal for matrix-heavy tasks in deep learning. In this lab, you'll compare training times and performance on CPU vs GPU and learn how to write GPU-efficient code.


##Check device availability

## TensorFlow

In [1]:
import tensorflow as tf
print("Is GPU available?", tf.config.list_physical_devices('GPU'))

Is GPU available? [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


##PyTorch

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


## 🚀 Moving Models and Data to GPU

To fully utilize the GPU, both the model and input data must be moved to the GPU device. This ensures the computation is performed on the GPU instead of the CPU.

Let's see how to do this in both TensorFlow and PyTorch.


##TensorFlow - Using GPU Automatically

In [3]:
# TensorFlow uses GPU by default when available
import tensorflow as tf

with tf.device('/GPU:0'):  # or '/CPU:0' for CPU
    a = tf.random.normal([1000, 1000])
    b = tf.random.normal([1000, 1000])
    c = tf.matmul(a, b)
    print("Operation completed on:", c.device)


Operation completed on: /job:localhost/replica:0/task:0/device:GPU:0


##PyTorch - Manual GPU Transfer

In [4]:
import torch

# Use 'cuda' if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example tensor operation on GPU
a = torch.randn(1000, 1000).to(device)
b = torch.randn(1000, 1000).to(device)
c = torch.matmul(a, b)

print("Tensor 'c' is on device:", c.device)


Tensor 'c' is on device: cuda:0


##Measuring Training Time on CPU vs GPU

## ⏱️ Performance Benchmark: CPU vs GPU

We'll train a simple model on the MNIST dataset using both CPU and GPU. This will help us visualize the speedup provided by GPU acceleration.

Steps:
- Train on CPU and measure the time
- Train on GPU and measure the time
- Compare the difference


In [5]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Data
transform = transforms.ToTensor()
train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# Model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)




100%|██████████| 9.91M/9.91M [00:00<00:00, 19.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 483kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.42MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.02MB/s]


##Train on CPU

In [6]:
# Train on CPU
def train_on_cpu():
    device = torch.device("cpu")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ CPU training time: {end_time - start_time:.2f} sec")

train_on_cpu()

✅ CPU training time: 10.06 sec


##Train on GPU
Change your runtine to T4 GPU and run the following code block

In [7]:
# Train on GPU
def train_on_gpu():
    if not torch.cuda.is_available():
        print("🚫 CUDA not available on this system.")
        return

    device = torch.device("cuda")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ GPU training time: {end_time - start_time:.2f} sec")

train_on_gpu()


✅ GPU training time: 9.03 sec


I suppose we got lesser time for the gpu, it makes more difference on larger models that have more number of layers and filters, gpu speeds up the matrix multiplication due to the presence of numerous small cores.

**So here's an activity for you**
##Use tensorflow to train a model on the MNIST digits dataset on both gpu and cpu and examine which one works faster.

Use your custom number of layers and filters to experiment with the hyperparameters of the model.

In [12]:
import time
import tensorflow as tf

print("Is GPU available?", tf.config.list_physical_devices('GPU'))


Is GPU available? [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [13]:
# 1. Load and preprocess the MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize to [0, 1] and add a channel dimension for Conv2D: (28, 28) -> (28, 28, 1)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train[..., tf.newaxis]
x_test = x_test[..., tf.newaxis]

print(x_train.shape, y_train.shape)


(60000, 28, 28, 1) (60000,)


In [14]:
# 2. Build a small CNN — change num_filters / num_conv_layers to experiment
def build_model(num_conv_layers=2, num_filters=32):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(28, 28, 1)))

    filters = num_filters
    for i in range(num_conv_layers):
        model.add(tf.keras.layers.Conv2D(filters, kernel_size=3, activation='relu'))
        model.add(tf.keras.layers.MaxPooling2D(pool_size=2))
        filters *= 2  # double filters each layer, like a typical CNN

    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(10, activation='softmax'))
    return model


In [15]:
# 3. Build a tf.data pipeline for batching/shuffling
batch_size = 64

def make_dataset():
    ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
    ds = ds.shuffle(buffer_size=10000).batch(batch_size)
    return ds


In [16]:
# 4. Generic training-loop function — pass in a device string like '/CPU:0' or '/GPU:0'
def train_on_device(device_name, num_conv_layers=2, num_filters=32, epochs=1):
    with tf.device(device_name):
        model = build_model(num_conv_layers=num_conv_layers, num_filters=num_filters)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
        train_dataset = make_dataset()

        start_time = time.time()
        for epoch in range(epochs):
            epoch_loss, num_batches = 0.0, 0
            for images, labels in train_dataset:
                with tf.GradientTape() as tape:
                    output = model(images, training=True)
                    loss = loss_fn(labels, output)
                gradients = tape.gradient(loss, model.trainable_variables)
                optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                epoch_loss += loss.numpy()
                num_batches += 1
            print(f"  Epoch {epoch+1}: avg loss = {epoch_loss/num_batches:.4f}")
        elapsed = time.time() - start_time

    return elapsed


In [17]:
# 5. Train on CPU
print("Training on CPU...")
cpu_time = train_on_device('/CPU:0', num_conv_layers=2, num_filters=32, epochs=1)
print(f"✅ TensorFlow CPU training time: {cpu_time:.2f} sec")


Training on CPU...
  Epoch 1: avg loss = 0.1543
✅ TensorFlow CPU training time: 95.39 sec


In [18]:
# 6. Train on GPU (change Colab runtime to GPU/T4 first, or run on a machine with CUDA)
if tf.config.list_physical_devices('GPU'):
    print("Training on GPU...")
    gpu_time = train_on_device('/GPU:0', num_conv_layers=2, num_filters=32, epochs=1)
    print(f"✅ TensorFlow GPU training time: {gpu_time:.2f} sec")
    print(f"\nSpeedup: {cpu_time / gpu_time:.2f}x faster on GPU")
else:
    print("🚫 No GPU available on this system — skipping GPU run.")


Training on GPU...
  Epoch 1: avg loss = 0.1559
✅ TensorFlow GPU training time: 57.27 sec

Speedup: 1.67x faster on GPU
